In [5]:
import os
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
persistent_directory = "db1/chroma_db"

# 1. Load Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Load DB
db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}  
)

# 3. Retrieve Documents (Reducing k to 3 to save tokens on free tier)
query = "What was the name of the autonomous spaceport drone ship that achieved the first successful sea landing?"
retriever = db.as_retriever(search_kwargs={"k": 3})
relevant_docs = retriever.invoke(query)

# 4. model 
model = ChatGroq(
    model_name="llama-3.3-70b-versatile", 
    temperature=0,
)

# 5. Prepare Input
combined_input = f"""Based on the following documents, answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in relevant_docs])}

If the answer isn't in the documents, say "I don't have enough information."
"""

messages = [
    SystemMessage(content="You are a helpful assistant specialized in Microsoft and GitHub history."),
    HumanMessage(content=combined_input),
]

# 6. Invoke and Print
try:
    result = model.invoke(messages)
    print("\n--- Generated Response (Gemini) ---")
    print('Answer: ',result.content)
except Exception as e:
    print(f"Error: {e}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1722.01it/s]



--- Generated Response (Gemini) ---
Answer:  The name of the autonomous spaceport drone ship that achieved the first successful sea landing is "Of Course I Still Love You".
